In [ ]:
# ============================================================
# INSTALLATION
# ============================================================

!pip -q install gdown openpyxl xlrd gradio requests ftfy


# ============================================================
# IMPORTS
# ============================================================

import os
import re
import io
import shutil
import requests
import pandas as pd
import gdown
import gradio as gr
import ftfy


# ============================================================
# GLOBAL STATE
# ============================================================

CURRENT_FILE_TYPE = None
CURRENT_FILE_NAME = "processed_file"
CURRENT_SHEETS = None


# ============================================================
# GOOGLE SHEETS DETECTION
# ============================================================

def extract_google_sheet_id(url):

    patterns = [
        r"docs\.google\.com/spreadsheets/d/([a-zA-Z0-9_-]+)",
        r"docs\.google\.com/spreadsheets/u/\d+/d/([a-zA-Z0-9_-]+)"
    ]

    for pattern in patterns:

        match = re.search(pattern, url)

        if match:
            return match.group(1)

    return None


# ============================================================
# GOOGLE DRIVE FILE ID
# ============================================================

def extract_drive_file_id(url):

    patterns = [
        r"drive\.google\.com/file/d/([a-zA-Z0-9_-]+)",
        r"drive\.google\.com/open\?id=([a-zA-Z0-9_-]+)",
        r"drive\.google\.com/uc\?.*id=([a-zA-Z0-9_-]+)"
    ]

    for pattern in patterns:

        match = re.search(pattern, url)

        if match:
            return match.group(1)

    return None


# ============================================================
# DOWNLOAD GOOGLE SHEETS AS XLSX
# ============================================================

def download_google_sheet(url):

    sheet_id = extract_google_sheet_id(url)

    if not sheet_id:
        raise ValueError(
            "لم أستطع استخراج Google Sheet ID من الرابط."
        )

    export_url = (
        f"https://docs.google.com/spreadsheets/d/"
        f"{sheet_id}/export?format=xlsx"
    )

    response = requests.get(
        export_url,
        timeout=60
    )

    if response.status_code != 200:

        raise ValueError(
            "تعذر تحميل Google Sheet. "
            "تأكد أن الملف متاح لمن يملك الرابط."
        )

    output_path = "/content/input_file.xlsx"

    with open(output_path, "wb") as f:
        f.write(response.content)

    return output_path


# ============================================================
# DOWNLOAD GOOGLE DRIVE FILE
# ============================================================

def download_google_drive_file(url):

    file_id = extract_drive_file_id(url)

    if not file_id:

        raise ValueError(
            "لم أستطع استخراج Google Drive File ID."
        )

    output_path = "/content/input_file"

    downloaded = gdown.download(
        id=file_id,
        output=output_path,
        quiet=False
    )

    if downloaded is None:

        raise ValueError(
            "تعذر تحميل الملف من Google Drive."
        )

    return downloaded


# ============================================================
# DIRECT URL DOWNLOAD
# ============================================================

def download_direct_url(url):

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    content_type = response.headers.get(
        "content-type",
        ""
    ).lower()

    if "excel" in content_type or "spreadsheet" in content_type:

        extension = ".xlsx"

    elif "csv" in content_type:

        extension = ".csv"

    else:

        extension = ".csv"

    output_path = f"/content/input_file{extension}"

    with open(output_path, "wb") as f:
        f.write(response.content)

    return output_path


# ============================================================
# FILE TYPE DETECTION
# ============================================================

def detect_file_type(path):

    # XLSX / XLSM / ZIP based Excel
    with open(path, "rb") as f:

        header = f.read(8)

    if header[:2] == b"PK":

        return "xlsx"

    # Old Excel format
    if header[:8] == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1":

        return "xls"

    # CSV / text
    return "csv"


# ============================================================
# FIX MOJIBAKE (ARABIC TEXT CORRUPTION FIX)
# ============================================================

def fix_mojibake_text(text):
    """
    إصلاح متقدم للنصوص العربية الفاسدة باستخدام ftfy والحلقات التكرارية
    """
    if pd.isna(text) or not isinstance(text, str):
        return text

    # المسار الأول: استخدام مكتبة ftfy القوية جداً في كشف وإصلاح الترميز
    fixed_text = ftfy.fix_text(text)

    # المسار الثاني: خوارزمية احتياطية للحالات المعقدة (Double Mojibake)
    # إذا كان النص لا يزال يحتوي على رموز فاسدة
    if 'Ø' in fixed_text or 'Ù' in fixed_text or 'Ã' in fixed_text:
        best_text = fixed_text
        # محاولة الإصلاح المتكرر حتى 3 مرات
        for _ in range(3):
            changed = False
            # تجربة الترميزات الأكثر شيوعاً للعربية
            for enc in ['cp1252', 'latin1', 'cp1256', 'iso-8859-6']:
                try:
                    decoded = best_text.encode(enc).decode('utf-8')
                    if decoded != best_text:
                        best_text = decoded
                        changed = True
                        break # إذا نجح الترميز، نعيد المحاولة من البداية
                except (UnicodeEncodeError, UnicodeDecodeError):
                    pass
            if not changed:
                break
        fixed_text = best_text

    return fixed_text

def fix_dataframe_mojibake(df):
    """
    يطبق إصلاح Mojibake على جميع أعمدة النصوص في الـ DataFrame
    """
    # إصلاح أسماء الأعمدة
    df.columns = [fix_mojibake_text(str(col)) for col in df.columns]

    # إصلاح القيم داخل الأعمدة النصية
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(fix_mojibake_text)

    return df


# ============================================================
# CSV READER
# ============================================================

def read_csv_safely(path):

    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp1256",
        "windows-1256",
        "iso-8859-6",
        "latin1"
    ]

    last_error = None

    for encoding in encodings:

        try:

            df = pd.read_csv(
                path,
                encoding=encoding
            )

            return df, encoding

        except Exception as error:

            last_error = error

    raise ValueError(
        "لم أستطع قراءة ملف CSV.\n"
        f"آخر خطأ: {last_error}"
    )


# ============================================================
# EXCEL READER
# ============================================================

def read_excel_file(path):

    excel_file = pd.ExcelFile(path)

    sheets = {}

    for sheet_name in excel_file.sheet_names:

        sheets[sheet_name] = pd.read_excel(
            path,
            sheet_name=sheet_name
        )

    return sheets


# ============================================================
# LOAD FILE
# ============================================================

def load_file(path):

    global CURRENT_FILE_TYPE
    global CURRENT_SHEETS

    file_type = detect_file_type(path)

    if file_type == "csv":

        df, encoding = read_csv_safely(path)

        # تطبيق إصلاح النص العربي
        df = fix_dataframe_mojibake(df)

        CURRENT_FILE_TYPE = "csv"
        CURRENT_SHEETS = None

        return df, f"CSV | Encoding: {encoding}"

    elif file_type in ["xlsx", "xls"]:

        sheets = read_excel_file(path)

        CURRENT_FILE_TYPE = file_type
        CURRENT_SHEETS = sheets

        first_sheet = list(sheets.keys())[0]
        df = sheets[first_sheet]

        # تطبيق إصلاح النص العربي
        df = fix_dataframe_mojibake(df)

        return (
            df,
            f"{file_type.upper()} | Sheets: {len(sheets)}"
        )

    else:

        raise ValueError(
            "نوع الملف غير مدعوم."
        )


# ============================================================
# SAVE FILE
# ============================================================

def save_file(df):

    global CURRENT_FILE_TYPE
    global CURRENT_SHEETS

    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    if CURRENT_FILE_TYPE == "csv":

        output_path = "/content/processed_file.csv"

        df.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path


    # --------------------------------------------------------
    # XLSX
    # --------------------------------------------------------

    elif CURRENT_FILE_TYPE == "xlsx":

        output_path = "/content/processed_file.xlsx"

        df.to_excel(
            output_path,
            index=False,
            engine="openpyxl"
        )

        return output_path


    # --------------------------------------------------------
    # XLS
    # --------------------------------------------------------

    elif CURRENT_FILE_TYPE == "xls":

        output_path = "/content/processed_file.xlsx"

        df.to_excel(
            output_path,
            index=False,
            engine="openpyxl"
        )

        return output_path


    raise ValueError(
        "نوع الملف غير معروف."
    )


# ============================================================
# MAIN PROCESS
# ============================================================

def process_link(link):

    global CURRENT_FILE_NAME

    if not link:

        raise gr.Error(
            "من فضلك ضع رابط الملف."
        )

    link = link.strip()

    try:

        # ----------------------------------------------------
        # Google Sheets
        # ----------------------------------------------------

        if extract_google_sheet_id(link):

            path = download_google_sheet(link)

            CURRENT_FILE_NAME = "google_sheet"

        # ----------------------------------------------------
        # Google Drive
        # ----------------------------------------------------

        elif extract_drive_file_id(link):

            path = download_google_drive_file(link)

            CURRENT_FILE_NAME = "drive_file"

        # ----------------------------------------------------
        # Direct URL
        # ----------------------------------------------------

        else:

            path = download_direct_url(link)

            CURRENT_FILE_NAME = "downloaded_file"


        # ----------------------------------------------------
        # Load
        # ----------------------------------------------------

        df, info = load_file(path)

        info = (
            f"تم تحميل الملف بنجاح\n\n"
            f"{info}\n"
            f"Rows: {len(df):,}\n"
            f"Columns: {len(df.columns):,}"
        )

        return df, info


    except Exception as error:

        raise gr.Error(
            f"حدث خطأ:\n\n{error}"
        )


# ============================================================
# SAVE
# ============================================================

def save_edited_data(df):

    if df is None:

        raise gr.Error(
            "لا توجد بيانات لحفظها."
        )

    try:

        df = pd.DataFrame(df)

        output_path = save_file(df)

        return output_path

    except Exception as error:

        raise gr.Error(
            f"حدث خطأ أثناء حفظ الملف:\n\n{error}"
        )


# ============================================================
# USER INTERFACE
# ============================================================

with gr.Blocks() as app:

    gr.Markdown(
        """
        # 📊 Arabic File Processor

        ألصق رابط الملف فقط، وسيتم اكتشاف نوعه تلقائياً.

        يدعم:
        - Google Sheets
        - Google Drive
        - CSV
        - XLSX
        - XLS
        """
    )


    link_input = gr.Textbox(
        label="File Link",
        placeholder="ألصق رابط الملف هنا..."
    )


    load_button = gr.Button(
        "📂 Load File"
    )


    file_info = gr.Textbox(
        label="File Information",
        interactive=False
    )


    data_editor = gr.Dataframe(
        label="Data",
        interactive=True,
        wrap=True
    )


    save_button = gr.Button(
        "💾 Save & Download"
    )


    output_file = gr.File(
        label="Your Modified File"
    )


    load_button.click(
        fn=process_link,
        inputs=link_input,
        outputs=[
            data_editor,
            file_info
        ]
    )


    save_button.click(
        fn=save_edited_data,
        inputs=data_editor,
        outputs=output_file
    )


app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://60eb8a82e16410c0eb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# INSTALLATION
# ============================================================

!pip -q install gdown openpyxl xlrd gradio requests ftfy


# ============================================================
# IMPORTS
# ============================================================

import os
import re
import io
import shutil
import requests
import pandas as pd
import gdown
import gradio as gr
import ftfy


# ============================================================
# GLOBAL STATE
# ============================================================

CURRENT_FILE_TYPE = None
CURRENT_FILE_NAME = "processed_file"
CURRENT_SHEETS = None


# ============================================================
# GOOGLE SHEETS DETECTION
# ============================================================

def extract_google_sheet_id(url):

    patterns = [
        r"docs\.google\.com/spreadsheets/d/([a-zA-Z0-9_-]+)",
        r"docs\.google\.com/spreadsheets/u/\d+/d/([a-zA-Z0-9_-]+)"
    ]

    for pattern in patterns:

        match = re.search(pattern, url)

        if match:
            return match.group(1)

    return None


# ============================================================
# GOOGLE DRIVE FILE ID
# ============================================================

def extract_drive_file_id(url):

    patterns = [
        r"drive\.google\.com/file/d/([a-zA-Z0-9_-]+)",
        r"drive\.google\.com/open\?id=([a-zA-Z0-9_-]+)",
        r"drive\.google\.com/uc\?.*id=([a-zA-Z0-9_-]+)"
    ]

    for pattern in patterns:

        match = re.search(pattern, url)

        if match:
            return match.group(1)

    return None


# ============================================================
# DOWNLOAD GOOGLE SHEETS AS XLSX
# ============================================================

def download_google_sheet(url):

    sheet_id = extract_google_sheet_id(url)

    if not sheet_id:
        raise ValueError(
            "لم أستطع استخراج Google Sheet ID من الرابط."
        )

    export_url = (
        f"https://docs.google.com/spreadsheets/d/"
        f"{sheet_id}/export?format=xlsx"
    )

    response = requests.get(
        export_url,
        timeout=60
    )

    if response.status_code != 200:

        raise ValueError(
            "تعذر تحميل Google Sheet. "
            "تأكد أن الملف متاح لمن يملك الرابط."
        )

    output_path = "/content/input_file.xlsx"

    with open(output_path, "wb") as f:
        f.write(response.content)

    return output_path


# ============================================================
# DOWNLOAD GOOGLE DRIVE FILE
# ============================================================

def download_google_drive_file(url):

    file_id = extract_drive_file_id(url)

    if not file_id:

        raise ValueError(
            "لم أستطع استخراج Google Drive File ID."
        )

    output_path = "/content/input_file"

    downloaded = gdown.download(
        id=file_id,
        output=output_path,
        quiet=False
    )

    if downloaded is None:

        raise ValueError(
            "تعذر تحميل الملف من Google Drive."
        )

    return downloaded


# ============================================================
# DIRECT URL DOWNLOAD
# ============================================================

def download_direct_url(url):

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    content_type = response.headers.get(
        "content-type",
        ""
    ).lower()

    if "excel" in content_type or "spreadsheet" in content_type:

        extension = ".xlsx"

    elif "csv" in content_type:

        extension = ".csv"

    else:

        extension = ".csv"

    output_path = f"/content/input_file{extension}"

    with open(output_path, "wb") as f:
        f.write(response.content)

    return output_path


# ============================================================
# FILE TYPE DETECTION
# ============================================================

def detect_file_type(path):

    with open(path, "rb") as f:

        header = f.read(8)

    if header[:2] == b"PK":

        return "xlsx"

    if header[:8] == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1":

        return "xls"

    return "csv"


# ============================================================
# FIX MOJIBAKE (ARABIC TEXT CORRUPTION FIX)
# ============================================================

def fix_mojibake_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return text

    fixed_text = ftfy.fix_text(text)

    if 'Ø' in fixed_text or 'Ù' in fixed_text or 'Ã' in fixed_text:
        best_text = fixed_text
        for _ in range(3):
            changed = False
            for enc in ['cp1252', 'latin1', 'cp1256', 'iso-8859-6']:
                try:
                    decoded = best_text.encode(enc).decode('utf-8')
                    if decoded != best_text:
                        best_text = decoded
                        changed = True
                        break
                except (UnicodeEncodeError, UnicodeDecodeError):
                    pass
            if not changed:
                break
        fixed_text = best_text

    return fixed_text

def fix_dataframe_mojibake(df):
    df.columns = [fix_mojibake_text(str(col)) for col in df.columns]

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(fix_mojibake_text)

    return df


# ============================================================
# CSV READER
# ============================================================

def read_csv_safely(path):

    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp1256",
        "windows-1256",
        "iso-8859-6",
        "latin1"
    ]

    last_error = None

    for encoding in encodings:

        try:

            df = pd.read_csv(
                path,
                encoding=encoding
            )

            return df, encoding

        except Exception as error:

            last_error = error

    raise ValueError(
        "لم أستطع قراءة ملف CSV.\n"
        f"آخر خطأ: {last_error}"
    )


# ============================================================
# EXCEL READER
# ============================================================

def read_excel_file(path):

    excel_file = pd.ExcelFile(path)

    sheets = {}

    for sheet_name in excel_file.sheet_names:

        sheets[sheet_name] = pd.read_excel(
            path,
            sheet_name=sheet_name
        )

    return sheets


# ============================================================
# LOAD FILE
# ============================================================

def load_file(path):

    global CURRENT_FILE_TYPE
    global CURRENT_SHEETS

    file_type = detect_file_type(path)

    if file_type == "csv":

        df, encoding = read_csv_safely(path)

        df = fix_dataframe_mojibake(df)

        CURRENT_FILE_TYPE = "csv"
        CURRENT_SHEETS = None

        return df, f"✅ تم قراءة ملف CSV بنجاح | الترميز: {encoding}"

    elif file_type in ["xlsx", "xls"]:

        sheets = read_excel_file(path)

        CURRENT_FILE_TYPE = file_type
        CURRENT_SHEETS = sheets

        first_sheet = list(sheets.keys())[0]
        df = sheets[first_sheet]

        df = fix_dataframe_mojibake(df)

        return (
            df,
            f"✅ تم قراءة ملف {file_type.upper()} بنجاح | عدد الأوراق: {len(sheets)}"
        )

    else:

        raise ValueError(
            "نوع الملف غير مدعوم."
        )


# ============================================================
# SAVE FILE
# ============================================================

def save_file(df):

    global CURRENT_FILE_TYPE
    global CURRENT_SHEETS

    if CURRENT_FILE_TYPE == "csv":

        output_path = "/content/processed_file.csv"

        df.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    elif CURRENT_FILE_TYPE in ["xlsx", "xls"]:

        output_path = "/content/processed_file.xlsx"

        df.to_excel(
            output_path,
            index=False,
            engine="openpyxl"
        )

        return output_path

    raise ValueError(
        "نوع الملف غير معروف."
    )


# ============================================================
# MAIN PROCESS
# ============================================================

def process_link(link):

    global CURRENT_FILE_NAME

    if not link:

        raise gr.Error(
            "من فضلك ضع رابط الملف."
        )

    link = link.strip()

    try:

        if extract_google_sheet_id(link):

            path = download_google_sheet(link)

            CURRENT_FILE_NAME = "google_sheet"

        elif extract_drive_file_id(link):

            path = download_google_drive_file(link)

            CURRENT_FILE_NAME = "drive_file"

        else:

            path = download_direct_url(link)

            CURRENT_FILE_NAME = "downloaded_file"

        df, info = load_file(path)

        return df, info, gr.File(value=None, visible=False)


    except Exception as error:

        raise gr.Error(
            f"حدث خطأ:\n\n{error}"
        )


# ============================================================
# SAVE
# ============================================================

def save_edited_data(df):

    if df is None:

        raise gr.Error(
            "لا توجد بيانات لحفظها."
        )

    try:

        df = pd.DataFrame(df)

        output_path = save_file(df)

        return gr.File(value=output_path, visible=True)

    except Exception as error:

        raise gr.Error(
            f"حدث خطأ أثناء حفظ الملف:\n\n{error}"
        )


# ============================================================
# USER INTERFACE
# ============================================================

theme = gr.themes.Soft(
    primary_hue="emerald",
    secondary_hue="blue",
    font=[gr.themes.GoogleFont("Cairo"), "sans-serif"]
)

with gr.Blocks() as app:

    gr.Markdown(
        """
        # 📊 مُصلح الملفات العربية (Arabic File Processor)
        **يقوم هذا التطبيق بإصلاح النصوص العربية المشوهة (Mojibake) في ملفات CSV و Excel تلقائياً.**

        ### 📝 طريقة الاستخدام:
        1. ألصق رابط الملف (Google Sheets، Google Drive، أو رابط مباشر) في الحقل أدناه.
        2. اضغط على زر **"تحميل ومعالجة الملف"**.
        3. انتظر قليلاً، سيتم عرض البيانات في الجدول بعد إصلاحها.
        4. يمكنك تعديل البيانات يدوياً من داخل الجدول إذا أردت.
        5. اضغط على زر **"حفظ الملف المصلح"** لتحميل النسخة النهائية.
        """
    )

    with gr.Row(variant="panel"):
        with gr.Column(scale=4):
            link_input = gr.Textbox(
                label="رابط الملف",
                placeholder="ألصق رابط Google Sheets أو Google Drive أو رابط مباشر هنا...",
                show_label=False,
                lines=1
            )
        with gr.Column(scale=1, min_width=200):
            load_button = gr.Button(
                "📂 تحميل ومعالجة الملف",
                variant="primary",
                size="lg"
            )

    file_info = gr.Textbox(
        label="حالة الملف",
        interactive=False,
        visible=False,
        show_label=False
    )

    gr.Markdown("### 📋 البيانات (يمكنك التعديل عليها مباشرة)")

    data_editor = gr.Dataframe(
        label="Data",
        interactive=True,
        wrap=True,
        show_label=False
    )

    with gr.Row(variant="panel"):
        with gr.Column(scale=1):
            save_button = gr.Button(
                "💾 حفظ الملف المصلح",
                variant="secondary",
                size="lg"
            )
        with gr.Column(scale=2):
            output_file = gr.File(
                label="📥 اضغط الرابط تحت لتحميل الملف الجاهز",
                visible=False,
                interactive=False
            )

    # ========================================================
    # EVENTS (ربط الأزرار بالوظائف)
    # ========================================================

    load_button.click(
        fn=process_link,
        inputs=link_input,
        outputs=[data_editor, file_info, output_file],
        show_progress="full"
    ).then(
        fn=lambda: gr.update(visible=True),
        outputs=file_info
    )

    save_button.click(
        fn=save_edited_data,
        inputs=data_editor,
        outputs=output_file,
        show_progress="full"
    )

# ============================================================
# LAUNCH APP (تمت إزالة title)
# ============================================================

app.launch(
    debug=True,
    theme=theme
)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3ad48d69388e517565.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
